In [1]:
# Importamos la biblioteca pandas para leer nuestros datos
# con el alias 'pd'
import pandas as pd

# Importamos la función train_test_split de sklearn.model_selection
# para dividir nuestros datos en conjuntos de entrenamiento y prueba
from sklearn.model_selection import train_test_split

# Importamos el StandardScaler para normalizar los datos.
from sklearn.preprocessing import StandardScaler

# Importamos el clasificador KNN de sklearn.neighbors
# para la clasificación y la regresión
from sklearn.neighbors import KNeighborsClassifier

# Importamos las funciones classification_report y confusion_matrix
# de sklearn.metrics para evaluar la precisión del modelo y
# para visualizar los resultados de la clasificación
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

from matplotlib import pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

# Establecemos la ruta donde se encuentran los datos
path = "/content/drive/MyDrive/Ciencia de datos/Datos TPI 2026/"

# Importamos los datos de los vuelos utilizando pandas
df1 = pd.read_excel(path + "Vuelos.xlsx")

# Importamos los datos de los nuevos vuelos utilizando pandas
df2 = pd.read_excel(path + "casosNuevos.xlsx")

df_clientes = pd.read_excel(path + "Clientes.xlsx")

Mounted at /content/drive


In [2]:
#importamos los paquetes que vamos a necesitar
from sklearn.tree import DecisionTreeClassifier #arbol de clasificación
from sklearn.model_selection import train_test_split #funcion para dividir datos en train y test
from sklearn.metrics import make_scorer, accuracy_score, recall_score, confusion_matrix
from sklearn.tree import plot_tree
import numpy as np
import seaborn as sns

#LA SINTAXIS EN PYTHON PARA APLICAR UN MODELO DE ÁRBOLES ES LA SIGUIENTE
#modelo = DecisionTreeClassifier(max_depth=N)
#modelo.fit(X_train, y_train)

#Como podemos ver, necesitamos separar las variables predictoras (X) de la variable predicha (y).
#Por otro lado, como en todo modelo predictivo, vamos a realizar una separación de los datos en entrenamiento (train) y prueba (test)


In [3]:
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_cliente                    50000 non-null  int64  
 1   sexo                          50000 non-null  object 
 2   provincia                     50000 non-null  object 
 3   edad                          50000 non-null  int64  
 4   ocupacion                     50000 non-null  object 
 5   cant_vuelos                   50000 non-null  int64  
 6   clase_preferida               50000 non-null  object 
 7   gasto_acumulado               50000 non-null  float64
 8   gasto_acumulado_extra         50000 non-null  float64
 9   programaMillas                50000 non-null  object 
 10  cantidad_millas               50000 non-null  int64  
 11  ingreso_mensual               50000 non-null  int64  
 12  anticipacion_compra_promedio  50000 non-null  int64  
 13  c

In [ ]:
df1.

In [ ]:
#DEFINIMOS LA FUNCIÓN one_hot_encoding para transformar las variables categóricas
def one_hot_encoding(df, categorical_features, final_columns=None, drop_originals=True):
  """
    Realiza codificación one-hot en las características categóricas especificadas de un DataFrame.
    Esta función aplica la codificación one-hot a la lista dada de características categóricas
    en el DataFrame utilizando el método get_dummies de pandas. Opcionalmente, puede eliminar
    las columnas categóricas originales del DataFrame. Además, permite asegurar que el DataFrame
    resultante incluya un conjunto específico de columnas, rellenando con ceros las que no estén presentes.

    Parámetros:
    - df (pandas.DataFrame): El DataFrame que contiene los datos.
    - categorical_features (list): Lista de columnas categóricas a codificar.
    - final_columns (list): Lista de columnas que deben estar presentes en el DataFrame final.
                            Se agregarán como columnas de ceros si no existen después de la codificación.
    - drop_originals (bool): Indica si se deben eliminar las columnas originales.
                             El valor predeterminado es True.

    Retorna:
    - pandas.DataFrame: DataFrame con las características categóricas codificadas y, opcionalmente,
                        sin las columnas categóricas originales. Incluye todas las columnas especificadas en final_columns.
    """

  # Realizar codificación one-hot
  encoded_features = pd.get_dummies(df[categorical_features], drop_first=True)
  df = pd.concat([df, encoded_features*1], axis=1)

  if drop_originals:
    df = df.drop(categorical_features, axis=1)

  # Asegurar que todas las columnas finales estén presentes
  if final_columns is not None:
    for column in final_columns:
      if column not in df.columns:
        df[column] = 0  # Añadir la columna faltante con ceros

  return df


In [ ]:
df1['temporada_alta'] = df1['temporada_alta'].astype(int)
df1['demora'] = df1['demora'].astype(int)
df1=df1.drop(columns=['id_vuelo'])

In [ ]:
#Aplicamos one hot encoding a nuestro dataframe
vars_cat = ['condiciones_climaticas','congestion_aerea']
df1_codificado = one_hot_encoding(df1, vars_cat)

In [ ]:
df1[(df1['distancia_vuelo'] > 10000) &
    (~df1['aeropuerto_origen'].isin(['EZE', 'MAD']))]

,id_vuelo,aeropuerto_origen,aeropuerto_destino,hora_salida_programada,dia_semana,distancia_vuelo,condiciones_climaticas,congestion_aerea,tipo_avion,ocupacion_vuelo,temporada_alta,puerta_embarque,visibilidad,tiempo_estimado_vuelo,demora


In [ ]:
df1[df1['ocupacion_vuelo'] == 0]

In [ ]:
df1[(df1['tiempo_estimado_vuelo'] > 10000) &
    (~df1['aeropuerto_origen'].isin(['EZE', 'MAD']))]

,id_vuelo,aeropuerto_origen,aeropuerto_destino,hora_salida_programada,dia_semana,distancia_vuelo,condiciones_climaticas,congestion_aerea,tipo_avion,ocupacion_vuelo,temporada_alta,puerta_embarque,visibilidad,tiempo_estimado_vuelo,demora


In [4]:
df1.describe()

,id_vuelo,distancia_vuelo,ocupacion_vuelo,puerta_embarque,visibilidad,tiempo_estimado_vuelo,demora
count,15000.00000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,20011.42920,1811.709667,71.381967,25.052200,15.371927,169.874467,0.318533
std,5791.68528,2276.650883,15.965463,14.130638,8.633404,171.010583,0.465923
min,10002.00000,35.000000,9.200000,1.000000,0.200000,22.000000,0.000000
25%,14993.75000,300.000000,61.075000,13.000000,8.100000,58.000000,0.000000
50%,20029.50000,960.000000,73.400000,25.000000,15.600000,103.000000,0.000000
75%,25041.25000,2620.000000,83.800000,37.000000,22.900000,220.000000,1.000000
max,30000.00000,10050.000000,99.800000,49.000000,30.000000,802.000000,1.000000


In [5]:
df_clientes.describe()

,id_cliente,edad,cant_vuelos,gasto_acumulado,gasto_acumulado_extra,cantidad_millas,ingreso_mensual,anticipacion_compra_promedio
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000
mean,25000.500000,37.827400,17.071860,6587.557023,953.732059,6759.361980,3062.882160,19.91066
std,14433.901067,11.479385,15.750424,12439.879256,1987.615263,23529.642402,3780.380841,21.40158
min,1.000000,18.000000,1.000000,71.250000,3.210000,0.000000,300.000000,0.00000
25%,12500.750000,29.000000,6.000000,1404.575000,167.480000,0.000000,1250.000000,5.00000
50%,25000.500000,38.000000,14.000000,3099.895000,403.255000,0.000000,2100.000000,13.00000
75%,37500.250000,46.000000,21.000000,6317.357500,912.547500,0.000000,3823.000000,27.00000
max,50000.000000,80.000000,99.000000,253140.320000,47394.970000,541589.000000,350075.000000,180.00000


In [ ]:
df1.info()

In [ ]:
# ==========================================
# 1. MAPAS DE CORRELACIÓN
# ==========================================

# Vuelos
plt.figure(figsize=(10, 8))
# Excluimos IDs y variables como 'puerta_embarque' que pasamos a categóricas antes
cols_vuelos_num = df1_codificado.select_dtypes(include=['int64', 'float64']).columns.drop('id_vuelo', errors='ignore')
corr_vuelos = df1_codificado[cols_vuelos_num].corr()
sns.heatmap(corr_vuelos, annot=True, cmap='viridis', fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlación - Variables Numéricas (Vuelos)")
plt.show()

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_vuelo                15000 non-null  int64  
 1   aeropuerto_origen       15000 non-null  object 
 2   aeropuerto_destino      15000 non-null  object 
 3   hora_salida_programada  15000 non-null  object 
 4   dia_semana              15000 non-null  object 
 5   distancia_vuelo         15000 non-null  int64  
 6   condiciones_climaticas  15000 non-null  object 
 7   congestion_aerea        15000 non-null  object 
 8   tipo_avion              15000 non-null  object 
 9   ocupacion_vuelo         15000 non-null  float64
 10  temporada_alta          15000 non-null  bool   
 11  puerta_embarque         15000 non-null  int64  
 12  visibilidad             15000 non-null  float64
 13  tiempo_estimado_vuelo   15000 non-null  int64  
 14  demora                  15000 non-null

In [ ]:
#Separamos en train y test, por un lado las variables predictoras (X) y por otro la predicha (y)
X_train, X_test, y_train, y_test = train_test_split(df1_codificado.drop(columns = ['demora','aeropuerto_origen','aeropuerto_destino','hora_salida_programada','tipo_avion','dia_semana','id_vuelo']),
                                                    df1_codificado.demora,
                                                    test_size=0.2,
                                                    random_state=17,
                                                    stratify=df1_codificado.demora)

In [ ]:
#CREAMOS EL MODELO CON LOS DATOS TRAIN
clf = DecisionTreeClassifier(max_depth=3)
clf.fit(X_train, y_train)

#CALCULAMOS LAS PREDICCIONES
y_pred = clf.predict(X_test)

#CALCULAMOS LA MATRIZ DE CONFUSION
conf_matrix = confusion_matrix(y_test, y_pred, labels = [0,1])

plt.figure(figsize=(5, 3))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Matriz de Confusión')
plt.xlabel('Predicciones')
plt.ylabel('Valores Verdaderos')

plt.show()

In [ ]:
#CALCULAMOS LA PRECISIÓN (ACCURACY) DEL MODELO
acc = accuracy_score(y_test, y_pred)
print(f"Precisión (Accuracy) del modelo: {acc:.4f}")
print(f"Es decir, el modelo acierta el {acc*100:.2f}% de las veces sobre el conjunto de prueba.")

In [ ]:
#GRAFICAMOS EL ARBOL
plt.figure(figsize=(26, 15))
plot_tree(clf, filled=True, class_names = np.unique(df1_codificado.demora).astype(str), feature_names=X_train.columns, rounded=True)
plt.show()